In [0]:
%pip install gdown

# Descarregar os dados da pasta do Google Drive
import gdown
import os

url_pasta = "https://drive.google.com/drive/u/0/folders/1UFC8kXGgggScPswXkYbOVLx6hULlrOT3"

# Criar volume no Unity Catalog para armazenar os dados
spark.sql("CREATE VOLUME IF NOT EXISTS retailcast_solo.default.raw_data")
caminho_local = "/Volumes/retailcast_solo/default/raw_data/gdrive_dados"

os.makedirs(caminho_local, exist_ok=True)

try:
    # Faz o download de todo o conteúdo da pasta
    gdown.download_folder(url=url_pasta, output=caminho_local, quiet=False, use_cookies=False)
except Exception as e:
    if "status code 401" in str(e) or "permission" in str(e).lower():
        raise PermissionError(
            "\n\n====== GOOGLE DRIVE PERMISSION ERROR ======\n"
            "The Google Drive folder is not publicly accessible.\n\n"
            "To fix this:\n"
            "1. Open the folder: https://drive.google.com/drive/u/0/folders/1UFC8kXGgggScPswXkYbOVLx6hULlrOT3\n"
            "2. Click the Share button (or right-click > Share)\n"
            "3. Change 'Restricted' to 'Anyone with the link'\n"
            "4. Set permission to 'Viewer'\n"
            "5. Click 'Done' and re-run this cell\n"
            "==========================================\n"
        ) from e
    else:
        raise

# 3. Ingerir os ficheiros CSV e criar tabelas Unity Catalog

# Definir os ficheiros e nomes das tabelas
ficheiros = {
    "rc_dados_vendas": "retailCast_dados_vendas.csv",
    "rc_eventos": "retailCast_eventos.csv",
    "rc_feriados": "retailCast_feriados.csv",
    "rc_info_loja": "retailCast_info_loja.csv"
}

# Criar cada tabela
for table_name, file_name in ficheiros.items():
    print(f"Processing {file_name} -> retailcast_solo.default.{table_name}")
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"{caminho_local}/{file_name}")
    
    # Gravar como tabela Unity Catalog (substitui se já existir)
    df.write.mode("overwrite").saveAsTable(f"retailcast_solo.default.{table_name}")
    
    print(f"✓ Table created: {table_name} ({df.count()} rows)")

print("\n✓ All tables created successfully!")
print("\nYou can now query them in SQL:")
print("  SELECT * FROM retailcast_solo.default.sales_data")
print("  SELECT * FROM retailcast_solo.default.events")
print("  SELECT * FROM retailcast_solo.default.holidays")
print("  SELECT * FROM retailcast_solo.default.store_info")

Retrieving folder contents


Processing file 1g7RpPn4kdHcepSoGe9DoHlSBRHU9VgIJ retailCast_dados_vendas.csv
Processing file 1XVCaSY5NdMrfuH9-8UJwr35_4rzcogkY retailCast_eventos.csv
Processing file 1qmzWjnNyyKBFhvb0ZNW7mxthhPfFmP5Z retailCast_feriados.csv
Processing file 1bompwPZ5cj76QwppSATme4y5A5gyakTm retailCast_info_loja.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1g7RpPn4kdHcepSoGe9DoHlSBRHU9VgIJ
To: /Volumes/retailcast_solo/default/raw_data/gdrive_dados/retailCast_dados_vendas.csv
100%|██████████| 9.99M/9.99M [00:01<00:00, 6.03MB/s]
Downloading...
From: https://drive.google.com/uc?id=1XVCaSY5NdMrfuH9-8UJwr35_4rzcogkY
To: /Volumes/retailcast_solo/default/raw_data/gdrive_dados/retailCast_eventos.csv
100%|██████████| 3.82k/3.82k [00:00<00:00, 15.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1qmzWjnNyyKBFhvb0ZNW7mxthhPfFmP5Z
To: /Volumes/retailcast_solo/default/raw_data/gdrive_dados/retailCast_feriados.csv
100%|██████████| 131k/131k [00:00<00:00, 1.60MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bompwPZ5cj76QwppSATme4y5A5gyakTm
To: /Volumes/retailcast_solo/default/raw_data/gdrive_dados/retailCast_info_loja.csv
100%|██████████| 9.03k/9.03k [00:00<00:00, 7.38MB/s]
Downl

Processing retailCast_dados_vendas.csv -> retailcast_solo.default.rc_dados_vendas
✓ Table created: rc_dados_vendas (98628 rows)
Processing retailCast_eventos.csv -> retailcast_solo.default.rc_eventos
✓ Table created: rc_eventos (65 rows)
Processing retailCast_feriados.csv -> retailcast_solo.default.rc_feriados
✓ Table created: rc_feriados (2271 rows)
Processing retailCast_info_loja.csv -> retailcast_solo.default.rc_info_loja
✓ Table created: rc_info_loja (70 rows)

✓ All tables created successfully!

You can now query them in SQL:
  SELECT * FROM retailcast_solo.default.sales_data
  SELECT * FROM retailcast_solo.default.events
  SELECT * FROM retailcast_solo.default.holidays
  SELECT * FROM retailcast_solo.default.store_info
